# Fruit Classification ANN — ReLU vs Leaky ReLU

This notebook contains an experiment comparing two activation functions, **ReLU** and **Leaky ReLU**, on an *Artificial Neural Network* (ANN) architecture for fruit image classification using the **Fruit360** dataset.

---

## 1. Install Library

Before running the notebook, make sure all required libraries are installed.

| Library | Purpose |
|---|---|
| `opencv-python` | Reading and processing images |
| `tqdm` | Progress bar during dataset loading |
| `scikit-image` | Additional image operations |
| `kagglehub` | Download datasets directly from Kaggle |
| `streamlit` | Building interactive web applications |

In [ ]:
!pip install -q opencv-python tqdm scikit-image kagglehub streamlit
print('All packages installed!')

---

## 2. Import Library

Importing all libraries that will be used throughout the notebook.

- **TensorFlow / Keras** → building and training the ANN model
- **NumPy** → array and matrix operations
- **Pandas** → tabular data manipulation
- **Matplotlib** → chart and graph visualization
- **OpenCV (cv2)** → image preprocessing
- **os / json** → file management and configuration storage
- **tqdm** → progress bar during dataset iteration
- **kagglehub** → downloading datasets from Kaggle

`np.random.seed(42)` and `tf.random.set_seed(42)` are set to ensure the experiment results are reproducible.

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import cv2
import json

from tqdm import tqdm

from tensorflow.keras.models import (
    Sequential,
    load_model
)

from tensorflow.keras.layers import (
    Dense,
    ReLU,
    LeakyReLU
)

from tensorflow.keras.utils import (
    to_categorical
)

import kagglehub

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow Version:", tf.__version__)

---

## 3. Download Dataset

The dataset used is **Fruit360** from Kaggle (`moltean/fruits`).

This dataset contains fruit images sized **100×100 pixels** with a white background, organized into *Training* and *Test* folders named after each fruit class.

`kagglehub.dataset_download()` will automatically download the dataset to the local cache directory.

In [ ]:
print("Downloading dataset...")

path = kagglehub.dataset_download(
    "moltean/fruits"
)

print("Dataset downloaded to:")
print(path)

---

## 4. Dataset Configuration

Defining the dataset configuration before the loading process:

- **`train_path`** and **`test_path`** → paths to the Training and Test folders inside the Fruit360 dataset
- **`classes`** → 10 fruit classes selected for this experiment
- **`IMG_SIZE = 64`** → all images will be resized to 64×64 pixels for memory efficiency and faster training
- **`INPUT_DIM = 64 × 64 × 3 = 12,288`** → total input neurons (the length of the vector after the image is *flattened*)

After the configuration is defined, the class list is saved to `classes.json` so it can be read by the Streamlit application.

In [ ]:
train_path = os.path.join(
    path,
    "fruits-360_100x100",
    "fruits-360",
    "Training"
)

test_path = os.path.join(
    path,
    "fruits-360_100x100",
    "fruits-360",
    "Test"
)

classes = [
    "Apple Red 1",
    "Banana 1",
    "Raspberry 3",
    "Avocado 1",
    "Kiwi 1",
    "Carrot 1",
    "Papaya 1",
    "Strawberry 1",
    "Pear 10",
    "Tomato 1"
]

IMG_SIZE = 64

INPUT_DIM = IMG_SIZE * IMG_SIZE * 3

print("Training Path :", train_path)
print("Test Path     :", test_path)
print("Classes       :", len(classes))
print("Input Dim     :", INPUT_DIM)

In [ ]:
# Save the class list to a JSON file
# This file will be read by the Streamlit app
# to ensure prediction label order is consistent with the training order
with open(
    "classes.json",
    "w"
) as f:

    json.dump(
        classes,
        f
    )

print("Classes saved!")

---

## 5. Preprocessing Function

The `preprocess_image()` function converts a raw image into a numerical vector ready to be fed into the model.

Preprocessing steps:

1. **Read image** → if the input is a file path, the image is read using `cv2.imread()` (in BGR format)
2. **Convert BGR → RGB** → OpenCV reads images in BGR format, while the model is trained with RGB, so conversion is necessary
3. **Resize to 64×64** → standardizes all image sizes so the input dimension is consistent
4. **Normalize (÷ 255)** → converts pixel values from the range [0, 255] to [0.0, 1.0] for more stable training and faster convergence
5. **Flatten** → converts the 2D array (64×64×3) into a 1D vector of length 12,288, since Dense layers require a vector as input

In [ ]:
def preprocess_image(
    img_input,
    is_path=False
):

    if is_path:

        img = cv2.imread(
            img_input,
            cv2.IMREAD_COLOR
        )

        if img is None:

            raise ValueError(
                f"Cannot read image: {img_input}"
            )

    else:

        img = img_input.copy()

    # Convert BGR (OpenCV format) to RGB
    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    # Resize to target dimensions
    img = cv2.resize(
        img,
        (IMG_SIZE, IMG_SIZE)
    )

    # Normalize pixel values to the range [0, 1]
    img = img.astype(
        "float32"
    ) / 255.0

    # Flatten into a 1D vector
    img = img.flatten()

    return img

---

## 6. Load Dataset Function

The `load_dataset()` function reads all images from the dataset folder and converts them into NumPy arrays.

How it works:
- Iterates through each fruit class (10 classes)
- For each class, reads all image files inside its folder
- Each image is processed with `preprocess_image()` and the result is appended to list `X`
- A numeric label (0–9) is appended to list `Y` based on the class index

If any image fails to load (corrupted or unsupported format), it is skipped without stopping the loading process.

The final result is returned as a tuple `(X, Y)` in NumPy array format.

In [ ]:
def load_dataset(folder_root):

    X = []
    Y = []

    for label, fruit in enumerate(classes):

        folder_path = os.path.join(
            folder_root,
            fruit
        )

        print(f"\nLoading {fruit}")

        for img_name in tqdm(
            os.listdir(folder_path)
        ):

            img_path = os.path.join(
                folder_path,
                img_name
            )

            try:

                img_flat = preprocess_image(
                    img_path,
                    is_path=True
                )

                X.append(img_flat)

                Y.append(label)

            except Exception as e:

                print(
                    f"Skip {img_name}: {e}"
                )

    return (
        np.array(X, dtype="float32"),
        np.array(Y, dtype="int32")
    )

---

## 7. Load Training & Test Dataset

Calls `load_dataset()` to load the training and test data into memory.

- **x_train / x_test** → flattened image arrays, shape: `(N, 12288)`
- **y_train / y_test** → numeric labels for each image, shape: `(N,)`

This process may take several minutes depending on the dataset size and machine specifications.

In [ ]:
# Load training dataset
print("Loading Training Dataset...")

x_train, y_train = load_dataset(
    train_path
)

# Load test dataset
print("\nLoading Test Dataset...")

x_test, y_test = load_dataset(
    test_path
)

In [ ]:
# Display dataset shape information
print("\nTrain Shape :", x_train.shape)
print("Test Shape  :", x_test.shape)

---

## 8. One Hot Encoding

The labels in `y_train` and `y_test` are currently integer values (0–9). A model with a *Softmax* output layer requires labels in **One Hot Encoding** format.

Conversion example:
- Label `3` (Avocado) → `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`
- Label `4` (Kiwi)    → `[0, 0, 0, 0, 1, 0, 0, 0, 0, 0]`

`to_categorical()` from Keras performs this conversion automatically. After conversion, the label shape changes from `(N,)` to `(N, 10)`.

In [ ]:
y_train = to_categorical(
    y_train,
    num_classes=len(classes)
)

y_test = to_categorical(
    y_test,
    num_classes=len(classes)
)

print(
    "Train Label Shape:",
    y_train.shape
)

print(
    "Test Label Shape:",
    y_test.shape
)

---

## 9. Build & Train ReLU Model

### Model Architecture

The first model uses **ReLU** (*Rectified Linear Unit*) as its activation function.

```
Input  : 12,288 neurons (64×64×3 pixels, flattened)
Dense  : 256 neurons → ReLU
Dense  : 128 neurons → ReLU
Dense  : 64  neurons → ReLU
Output : 10  neurons → Softmax (one per fruit class)
```

**Why ReLU?**
ReLU returns 0 for negative inputs and the original value for positive inputs: `f(x) = max(0, x)`. It is computationally efficient, helps address the *vanishing gradient* problem, and generally performs well in deep networks.

In [ ]:
# Build the ReLU model architecture
relu_model = Sequential([

    Dense(
        256,
        input_shape=(INPUT_DIM,)
    ),

    ReLU(),

    Dense(128),

    ReLU(),

    Dense(64),

    ReLU(),

    Dense(
        len(classes),
        activation="softmax"
    )

], name="ReLU_Model")

### Compile ReLU Model

Before training, the model must be *compiled* by specifying:

- **Optimizer `adam`** → an adaptive optimization algorithm that combines *momentum* and *RMSProp*. Adam is widely used because it converges faster than standard SGD
- **Loss `categorical_crossentropy`** → the standard loss function for multi-class classification with One Hot Encoded labels
- **Metrics `accuracy`** → the evaluation metric displayed during training

In [ ]:
# Compile the ReLU model
relu_model.compile(

    optimizer="adam",

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

relu_model.summary()

### Train ReLU Model

The model is trained for **30 epochs** with a **batch size of 32**.

- **epoch** → one complete pass through the entire training dataset
- **batch_size** → the number of samples processed before the model updates its weights
- **validation_data** → the test data is used to monitor model performance at each epoch, enabling detection of *overfitting*

In [ ]:
# Train the ReLU model
history_relu = relu_model.fit(

    x_train,
    y_train,

    validation_data=(
        x_test,
        y_test
    ),

    epochs=30,

    batch_size=32,

    verbose=1
)

### Evaluate ReLU Model

After training is complete, the model is evaluated on the test data to obtain the final **accuracy** and **loss** values.

In [ ]:
# Evaluate the ReLU model on the test data
relu_loss, relu_acc = relu_model.evaluate(
    x_test,
    y_test,
    verbose=0
)

print(
    "ReLU Accuracy:",
    relu_acc
)

In [ ]:
# Save the full ReLU model (architecture + weights)
relu_model.save(
    "fruit_relu_model.keras"
)

print(
    "ReLU model saved!"
)

---

## 10. Build & Train Leaky ReLU Model

### Model Architecture

The second model uses **Leaky ReLU** as its activation function with `negative_slope = 0.1`.

```
Input  : 12,288 neurons
Dense  : 256 neurons → Leaky ReLU (α=0.1)
Dense  : 128 neurons → Leaky ReLU (α=0.1)
Dense  : 64  neurons → Leaky ReLU (α=0.1)
Output : 10  neurons → Softmax
```

**Difference from ReLU:**
ReLU outputs 0 for all negative inputs, which can cause neurons to "die" (*dying ReLU problem*) if weights consistently produce negative values. Leaky ReLU addresses this by allowing a small gradient for negative values: `f(x) = x` if `x > 0`, and `f(x) = α·x` if `x ≤ 0`. With `α = 0.1`, negative inputs are still passed through at 10% of their original scale.

In [ ]:
# Build the Leaky ReLU model architecture
leaky_model = Sequential([

    Dense(
        256,
        input_shape=(INPUT_DIM,)
    ),

    LeakyReLU(
        negative_slope=0.1
    ),

    Dense(128),

    LeakyReLU(
        negative_slope=0.1
    ),

    Dense(64),

    LeakyReLU(
        negative_slope=0.1
    ),

    Dense(
        len(classes),
        activation="softmax"
    )

], name="LeakyReLU_Model")

In [ ]:
# Compile the Leaky ReLU model
# Same configuration as ReLU to ensure a fair comparison
leaky_model.compile(

    optimizer="adam",

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

leaky_model.summary()

In [ ]:
# Train the Leaky ReLU model
# Same epochs and batch size as ReLU for a valid comparison
history_leaky = leaky_model.fit(

    x_train,
    y_train,

    validation_data=(
        x_test,
        y_test
    ),

    epochs=30,

    batch_size=32,

    verbose=1
)

In [ ]:
# Evaluate the Leaky ReLU model on the test data
leaky_loss, leaky_acc = leaky_model.evaluate(
    x_test,
    y_test,
    verbose=0
)

print(
    "Leaky ReLU Accuracy:",
    leaky_acc
)

---

## 11. Model Comparison

Comparing the performance of both models based on **accuracy** and **loss** on the test data.

Winner determination logic:
1. The model with **higher accuracy** wins directly
2. If accuracy is exactly equal, the model with **lower loss** wins (lower loss = more confident and precise predictions)

In [ ]:
print("="*50)

print(
    f"{'Model':<15}"
    f"{'Accuracy':>12}"
    f"{'Loss':>12}"
)

print("="*50)

print(
    f"{'ReLU':<15}"
    f"{relu_acc:>12.4f}"
    f"{relu_loss:>12.6f}"
)

print(
    f"{'Leaky ReLU':<15}"
    f"{leaky_acc:>12.4f}"
    f"{leaky_loss:>12.6f}"
)

print("="*50)

# Determine the winner based on accuracy, with loss as a tiebreaker
if relu_acc > leaky_acc:

    winner = "ReLU"

elif leaky_acc > relu_acc:

    winner = "Leaky ReLU"

else:

    winner = (
        "ReLU"
        if relu_loss < leaky_loss
        else "Leaky ReLU"
    )

print("Winner :", winner)

---

## 12. Accuracy & Loss Curve

Visualizing the training and validation curves for both models over 30 epochs.

These curves are important for detecting:
- **Underfitting** → both training and validation accuracy are consistently low
- **Overfitting** → training accuracy is high but validation accuracy is significantly lower (the model has memorized the training data)
- **Normal convergence** → both curves approach a stable value together

Four graphs are displayed: Accuracy and Loss for each model (ReLU & Leaky ReLU).

In [ ]:
fig, axes = plt.subplots(
    2,
    2,
    figsize=(14,10)
)

# ReLU Accuracy
axes[0,0].plot(
    history_relu.history['accuracy'],
    label='Train'
)

axes[0,0].plot(
    history_relu.history['val_accuracy'],
    label='Validation'
)

axes[0,0].set_title(
    'ReLU Accuracy'
)

axes[0,0].legend()

# ReLU Loss
axes[0,1].plot(
    history_relu.history['loss'],
    label='Train'
)

axes[0,1].plot(
    history_relu.history['val_loss'],
    label='Validation'
)

axes[0,1].set_title(
    'ReLU Loss'
)

axes[0,1].legend()

# Leaky ReLU Accuracy
axes[1,0].plot(
    history_leaky.history['accuracy'],
    label='Train'
)

axes[1,0].plot(
    history_leaky.history['val_accuracy'],
    label='Validation'
)

axes[1,0].set_title(
    'Leaky ReLU Accuracy'
)

axes[1,0].legend()

# Leaky ReLU Loss
axes[1,1].plot(
    history_leaky.history['loss'],
    label='Train'
)

axes[1,1].plot(
    history_leaky.history['val_loss'],
    label='Validation'
)

axes[1,1].set_title(
    'Leaky ReLU Loss'
)

axes[1,1].legend()

plt.tight_layout()
plt.show()

---

## 13. Confusion Matrix — ReLU

A **Confusion Matrix** displays the detailed prediction results of a model in an N×N table (N = number of classes).

- **Rows** → *actual* labels (ground truth)
- **Columns** → *predicted* labels (model output)
- **Diagonal** → correct predictions (*True Positives*)
- **Off-diagonal** → misclassifications

The Confusion Matrix helps identify which class pairs the model frequently confuses — information that cannot be obtained from the overall accuracy score alone.

In [ ]:
!pip install seaborn

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

In [ ]:
# Convert One Hot Encoded labels back to integers for the confusion matrix
y_true = np.argmax(
    y_test,
    axis=1
)

# Get ReLU model predictions on the entire test set
y_pred_relu = np.argmax(
    relu_model.predict(x_test),
    axis=1
)

cm_relu = confusion_matrix(
    y_true,
    y_pred_relu
)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm_relu,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title(
    'Confusion Matrix - ReLU'
)

plt.xlabel(
    'Predicted'
)

plt.ylabel(
    'Actual'
)

plt.show()

---

## 14. Confusion Matrix — Leaky ReLU

Confusion Matrix for the Leaky ReLU model. Compare this with the ReLU confusion matrix above to observe differences in misclassification patterns between the two models.

In [ ]:
# Get Leaky ReLU model predictions on the entire test set
y_pred_leaky = np.argmax(
    leaky_model.predict(x_test),
    axis=1
)

cm_leaky = confusion_matrix(
    y_true,
    y_pred_leaky
)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm_leaky,
    annot=True,
    fmt='d',
    cmap='Greens'
)

plt.title(
    'Confusion Matrix - Leaky ReLU'
)

plt.xlabel(
    'Predicted'
)

plt.ylabel(
    'Actual'
)

plt.show()

---

## 15. Test Single Prediction

Tests the predictions of both models on a **single random image** from the test data.

An image is randomly selected from `x_test`, and its actual label is compared against the predictions from both the ReLU and Leaky ReLU models. This provides a visual sanity check to confirm the model is classifying images correctly.

In [ ]:
# Pick a random image from the test set
test_idx = np.random.randint(
    0,
    len(x_test)
)

test_img_flat = x_test[test_idx]

actual_label = np.argmax(
    y_test[test_idx]
)

# Display the image (reshape from vector back to 2D image)
plt.figure(figsize=(4,4))

plt.imshow(
    test_img_flat.reshape(
        IMG_SIZE,
        IMG_SIZE,
        3
    )
)

plt.title(
    f'Actual: {classes[actual_label]}'
)

plt.axis('off')

plt.show()

In [ ]:
# Run predictions from both models on the same image
inp = np.expand_dims(
    test_img_flat,
    axis=0
)

relu_pred = relu_model.predict(
    inp,
    verbose=0
)

leaky_pred = leaky_model.predict(
    inp,
    verbose=0
)

relu_class = np.argmax(
    relu_pred
)

leaky_class = np.argmax(
    leaky_pred
)

print("="*50)
print(
    "TEST PREDICTION RESULT"
)
print("="*50)
print(
    f"Actual Label      : {classes[actual_label]}"
)
print(
    f"ReLU Prediction   : {classes[relu_class]}"
)
print(
    f"Leaky Prediction  : {classes[leaky_class]}"
)

---

## 16. Prediction Comparison Table

Compares the predictions of both models on **20 random images** from the test set.

This table directly shows on which samples ReLU and Leaky ReLU agree or disagree, and whether either prediction matches the actual label.

In [ ]:
print(
    "\n===== PREDICTION COMPARISON =====\n"
)

print(
    f"{'No':<5}"
    f"{'Actual':<20}"
    f"{'ReLU':<20}"
    f"{'Leaky':<20}"
)

print("-"*80)

# Select 20 random indices from the test set (without replacement)
sample_idx = np.random.choice(
    len(x_test),
    20,
    replace=False
)

for idx, i in enumerate(sample_idx):

    inp = np.expand_dims(
        x_test[i],
        axis=0
    )

    actual = np.argmax(
        y_test[i]
    )

    relu_pred = np.argmax(
        relu_model.predict(
            inp,
            verbose=0
        )
    )

    leaky_pred = np.argmax(
        leaky_model.predict(
            inp,
            verbose=0
        )
    )

    print(
        f"{idx+1:<5}"
        f"{classes[actual]:<20}"
        f"{classes[relu_pred]:<20}"
        f"{classes[leaky_pred]:<20}"
    )

---

## 17. Final Result & Save Best Model

Displays a summary of the final experiment results and determines the best model based on accuracy (with loss as a tiebreaker).

The best model's weights are saved to a `.weights.h5` file so they can be loaded by the Streamlit application without needing to retrain from scratch.

In [ ]:
print("ReLU Accuracy :", relu_acc)
print("ReLU Loss     :", relu_loss)

print("Leaky Accuracy:", leaky_acc)
print("Leaky Loss    :", leaky_loss)

In [ ]:
# Display a summary table and determine the final winner
print("="*50)
print(
    f"{'Model':<15}"
    f"{'Accuracy':>12}"
    f"{'Loss':>12}"
)
print("="*50)
print(
    f"{'ReLU':<15}"
    f"{relu_acc:>12.4f}"
    f"{relu_loss:>12.6f}"
)
print(
    f"{'Leaky ReLU':<15}"
    f"{leaky_acc:>12.4f}"
    f"{leaky_loss:>12.6f}"
)
print("="*50)

# Winner: higher accuracy wins
# If accuracy is tied, lower loss wins
if relu_acc > leaky_acc:
    winner = "ReLU"
elif leaky_acc > relu_acc:
    winner = "Leaky ReLU"
else:
    if relu_loss < leaky_loss:
        winner = "ReLU"
    else:
        winner = "Leaky ReLU"

print(f"\n🏆 WINNER : {winner}")

In [ ]:
# Save the ReLU model weights (winner) to an .h5 file
# This file will be loaded by the Streamlit application
relu_model.save_weights(
    "fruit_best_weights.weights.h5"
)

print("Weights saved!")

---

## 18. Conclusion

This experiment compared the performance of an ANN using **ReLU** and **Leaky ReLU** activation functions for fruit image classification on the Fruit360 dataset. Both models were trained with identical architecture and configuration to ensure a fair comparison.

**Experiment Results:**

| Model | Accuracy | Loss |
|---|---|---|
| ReLU | **98.34%** | 0.054951 |
| Leaky ReLU | 97.92% | **0.039673** |

ReLU outperformed in terms of **accuracy** and was therefore selected as the best model, despite Leaky ReLU producing a lower loss value.

**Limitations:**
A plain ANN operates on flattened raw pixels and cannot effectively capture spatial features such as shape, texture, and edges. This limits the model's generalization capability on images outside the dataset distribution (e.g., fruit photos with different backgrounds or lighting conditions).

**Recommendations for Future Work:**
- Use a **CNN** (*Convolutional Neural Network*) for better spatial feature extraction
- Apply **data augmentation** to improve generalization
- Consider **EarlyStopping** and **ModelCheckpoint** callbacks for more efficient training